# Consultas

## Conexión

In [1]:
import os
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()   # la contraseña está en el .env como PASS_SQL
PASS = os.getenv("PASS_SQL")

In [2]:
def url(bd):
    return URL.create("postgresql+psycopg2", username="postgres", password=PASS,
                      host="localhost", port=5432, database=bd)

# Creamos la base de datos si no existe (hay que conectarse a la de por defecto, "postgres")
admin = create_engine(url("postgres"), isolation_level="AUTOCOMMIT")
with admin.connect() as conn:
    existe = conn.execute(text("SELECT 1 FROM pg_database WHERE datname = 'pipeline_deudas'")).scalar()
    if not existe:
        conn.execute(text("CREATE DATABASE pipeline_deudas"))

engine = create_engine(url("pipeline_deudas"))

In [3]:
# Atajo para lanzar consultas y que salgan como DataFrame
def q(sql):
    return pd.read_sql(text(sql), engine)

q("SELECT version()")

,version
0,"PostgreSQL 18.6 on x86_64-windows, compiled by..."


## Tablas

In [9]:
# Sin clave foránea entre expedientes y clientes porque hay expedientes con un cliente que no está en clientes
ddl = [
    "DROP TABLE IF EXISTS expedientes",
    "DROP TABLE IF EXISTS clientes",
    "DROP TABLE IF EXISTS oficinas",
    """
    CREATE TABLE oficinas (
        nombre_oficina          TEXT PRIMARY KEY,
        responsable             TEXT,
        objetivo_conversion_pct NUMERIC(4,1),
        fecha_apertura_oficina  DATE
    )""",
    """
    CREATE TABLE clientes (
        cliente_id          INT PRIMARY KEY,
        nombre_completo     TEXT,
        fecha_entrada       DATE,
        canal_captacion     TEXT,
        departamento_oficina TEXT,
        codigo_postal       CHAR(5),
        email               TEXT,
        telefono            CHAR(9),
        ingresos_mensuales  NUMERIC(10,2),
        tiene_aval          BOOLEAN
    )""",
    """
    CREATE TABLE expedientes (
        expediente_id           INT PRIMARY KEY,
        cliente_id              INT,
        fecha_apertura          DATE,
        deuda_total             NUMERIC(12,2),
        num_acreedores          INT,
        acreedores              TEXT,
        cuota_mensual_propuesta NUMERIC(10,2),
        estado_expediente       TEXT
    )""",
]

with engine.begin() as conn:
    for s in ddl:
        conn.execute(text(s))

## Carga de los CSV

In [24]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE expedientes, clientes, oficinas"))

CLEAN = Path("..") / "clean"

def a_snake(col):
    # ClienteId -> cliente_id
    return re.sub(r"(?<!^)(?=[A-Z])", "_", col).lower()

def cargar(archivo, tabla, **kwargs):
    df = pd.read_csv(CLEAN / archivo, sep=";", **kwargs)
    df.columns = [a_snake(c) for c in df.columns]
    df.to_sql(tabla, engine, if_exists="append", index=False)
    return len(df)

# Teléfono y CP como texto para no perder los ceros. Si falla por clave duplicada, mirar los duplicados en el EDA
print(cargar("oficinas_clean.csv", "oficinas", parse_dates=["FechaAperturaOficina"]))
print(cargar("clientes_clean.csv", "clientes",
             dtype={"CodigoPostal": str, "Telefono": str, "TieneAval": "boolean"},
             parse_dates=["FechaEntrada"]))
print(cargar("expedientes_clean.csv", "expedientes",
             dtype={"ClienteId": "Int64", "NumAcreedores": "Int64"},
             parse_dates=["FechaApertura"]))

5
800
1000


## Consultas

Primero un vistazo

In [11]:
q("""
SELECT 'clientes' AS tabla, COUNT(*) AS filas FROM clientes
UNION ALL SELECT 'expedientes', COUNT(*) FROM expedientes
UNION ALL SELECT 'oficinas', COUNT(*) FROM oficinas
""")

,tabla,filas
0,clientes,800
1,expedientes,1000
2,oficinas,5


In [12]:
q("SELECT * FROM clientes LIMIT 5")

,cliente_id,nombre_completo,fecha_entrada,canal_captacion,departamento_oficina,codigo_postal,email,telefono,ingresos_mensuales,tiene_aval
0,614,"Domínguez Gómez, Francisco",2025-04-26,Recomendación,Valencia,46887,f.dominguez63@example.com,672949241,2141.38,True
1,392,Marta García Pérez,2025-02-23,Redes Sociales,Madrid,28971,m.garcia68@example.org,686083774,NaN,False
2,82,María Díaz López,2025-05-12,Seo Orgánico,Málaga,29178,m.diaz3@example.com,623559456,1101.00,None
3,476,"Torres Navarro, Ana",2025-08-24,Google Ads,Valencia,46608,a.torres36@example.org,638182902,2773.00,True
4,449,Díaz Romero Lucía,2025-06-06,Teléfono / Directo,Madrid,28834,l.diaz8@example.org,NaN,2975.06,True


Expedientes por estado

In [13]:
q("""
SELECT estado_expediente,
       COUNT(*) AS expedientes,
       ROUND(AVG(deuda_total), 0) AS deuda_media
FROM expedientes
GROUP BY estado_expediente
ORDER BY expedientes DESC
""")

,estado_expediente,expedientes,deuda_media
0,Plan Activo,337,36415.0
1,Aceptado / En Negociación,291,37269.0
2,En Estudio,199,140764.0
3,Cancelado Con Éxito,96,34867.0
4,Incumplido,57,33032.0
5,NaN,20,37312.0


Deuda por oficina (join con clientes)

In [14]:
q("""
SELECT c.departamento_oficina,
       COUNT(*) AS expedientes,
       SUM(e.deuda_total) AS deuda_total
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
GROUP BY c.departamento_oficina
ORDER BY deuda_total DESC
""")

,departamento_oficina,expedientes,deuda_total
0,Barcelona,222,17796076.52
1,Sevilla,177,16227332.01
2,Valencia,188,6760901.89
3,Málaga,178,6335913.90
4,Madrid,180,6246706.32
5,NaN,29,916645.38


Lo mismo pero con el responsable de la oficina (3 tablas)

In [15]:
q("""
SELECT o.nombre_oficina, o.responsable, o.objetivo_conversion_pct,
       COUNT(DISTINCT c.cliente_id) AS clientes,
       COUNT(e.expediente_id) AS expedientes,
       SUM(e.deuda_total) AS deuda_total
FROM oficinas o
LEFT JOIN clientes c ON c.departamento_oficina = o.nombre_oficina
LEFT JOIN expedientes e ON e.cliente_id = c.cliente_id
GROUP BY o.nombre_oficina, o.responsable, o.objetivo_conversion_pct
ORDER BY deuda_total DESC
""")

,nombre_oficina,responsable,objetivo_conversion_pct,clientes,expedientes,deuda_total
0,Barcelona,Laura Torres,28.0,164,222,17796076.52
1,Sevilla,Ana Ruiz,35.0,158,177,16227332.01
2,Valencia,Javier Díaz,32.0,149,188,6760901.89
3,Málaga,Marta Navarro,30.0,158,178,6335913.90
4,Madrid,Pedro Gil,30.0,147,180,6246706.32


Canal de captación, deuda media

In [16]:
q("""
SELECT c.canal_captacion,
       COUNT(*) AS expedientes,
       ROUND(AVG(e.deuda_total), 0) AS deuda_media
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
GROUP BY c.canal_captacion
ORDER BY deuda_media DESC
""")

,canal_captacion,expedientes,deuda_media
0,Redes Sociales,272,112214.0
1,NaN,26,44543.0
2,Seo Orgánico,170,37596.0
3,Google Ads,304,36275.0
4,Recomendación,91,34518.0
5,Teléfono / Directo,111,34395.0


Top 3 expedientes con más deuda de cada oficina (función de ventana)

In [17]:
q("""
SELECT *
FROM (
    SELECT c.departamento_oficina, e.expediente_id, e.deuda_total,
           ROW_NUMBER() OVER (PARTITION BY c.departamento_oficina ORDER BY e.deuda_total DESC) AS puesto
    FROM expedientes e
    JOIN clientes c ON c.cliente_id = e.cliente_id
    WHERE e.deuda_total IS NOT NULL
) t
WHERE puesto <= 3
ORDER BY departamento_oficina, puesto
""")

,departamento_oficina,expediente_id,deuda_total,puesto
0,Barcelona,661,9999999.00,1
1,Barcelona,484,64813.00,2
2,Barcelona,529,64728.00,3
3,Madrid,729,64520.00,1
4,Madrid,638,64421.08,2
5,Madrid,19,63703.00,3
6,Málaga,197,64552.00,1
7,Málaga,72,64316.00,2
8,Málaga,953,64241.00,3
9,Sevilla,276,9999999.00,1


Expedientes por mes, acumulado y diferencia con el mes anterior

In [18]:
q("""
WITH mensual AS (
    SELECT DATE_TRUNC('month', fecha_apertura)::date AS mes, COUNT(*) AS n
    FROM expedientes
    WHERE fecha_apertura IS NOT NULL
    GROUP BY 1
)
SELECT mes, n,
       SUM(n) OVER (ORDER BY mes) AS acumulado,
       n - LAG(n) OVER (ORDER BY mes) AS dif_mes_anterior
FROM mensual
ORDER BY mes
""")

,mes,n,acumulado,dif_mes_anterior
0,2024-11-01,1,1.0,NaN
1,2024-12-01,3,4.0,2.0
2,2025-01-01,42,46.0,39.0
3,2025-02-01,72,118.0,30.0
4,2025-03-01,97,215.0,25.0
5,2025-04-01,95,310.0,-2.0
6,2025-05-01,102,412.0,7.0
7,2025-06-01,109,521.0,7.0
8,2025-07-01,94,615.0,-15.0
9,2025-08-01,88,703.0,-6.0


% de incumplidos por oficina

In [19]:
q("""
SELECT c.departamento_oficina,
       COUNT(*) AS expedientes,
       SUM(CASE WHEN e.estado_expediente = 'Incumplido' THEN 1 ELSE 0 END) AS incumplidos,
       ROUND(100.0 * SUM(CASE WHEN e.estado_expediente = 'Incumplido' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_incumplidos
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
GROUP BY c.departamento_oficina
ORDER BY pct_incumplidos DESC
""")

,departamento_oficina,expedientes,incumplidos,pct_incumplidos
0,NaN,29,2,6.9
1,Valencia,188,13,6.9
2,Málaga,178,11,6.2
3,Sevilla,177,10,5.6
4,Barcelona,222,12,5.4
5,Madrid,180,5,2.8


Cuota propuesta respecto a los ingresos (los que se llevan más de la mitad)

In [20]:
q("""
SELECT e.expediente_id, c.cliente_id, c.ingresos_mensuales, e.cuota_mensual_propuesta,
       ROUND(100.0 * e.cuota_mensual_propuesta / c.ingresos_mensuales, 1) AS pct_cuota
FROM expedientes e
JOIN clientes c ON c.cliente_id = e.cliente_id
WHERE c.ingresos_mensuales > 0
  AND e.cuota_mensual_propuesta > 0.5 * c.ingresos_mensuales
ORDER BY pct_cuota DESC
LIMIT 10
""")

,expediente_id,cliente_id,ingresos_mensuales,cuota_mensual_propuesta,pct_cuota
0,562,222,8.44,401.00,4751.2
1,489,259,19.00,614.00,3231.6
2,559,259,19.00,506.00,2663.2
3,919,612,42.00,657.00,1564.3
4,366,570,48.00,697.00,1452.1
5,972,124,25.00,291.00,1164.0
6,119,186,36.39,396.28,1089.0
7,177,186,36.39,393.08,1080.2
8,257,612,42.00,445.00,1059.5
9,64,300,30.00,266.00,886.7


Clientes sin ningún expediente

In [21]:
q("""
SELECT c.cliente_id, c.nombre_completo, c.departamento_oficina
FROM clientes c
LEFT JOIN expedientes e ON e.cliente_id = c.cliente_id
WHERE e.expediente_id IS NULL
""")

,cliente_id,nombre_completo,departamento_oficina
0,251,David Navarro,Barcelona
1,120,Ana Martínez Romero,Valencia
2,681,"Torres Ruiz, David",Sevilla
3,647,Francisco Moreno Jiménez,Málaga
4,452,"Sánchez Martínez, Manuel",Barcelona
...,...,...,...
282,240,Lucía Romero Pérez,Málaga
283,772,Romero Sánchez Javier,Madrid
284,740,"López Domínguez, Pedro",Valencia
285,616,García Moreno David,Valencia


Expedientes con un cliente que no existe en clientes

In [22]:
q("""
SELECT e.expediente_id, e.cliente_id, e.deuda_total
FROM expedientes e
LEFT JOIN clientes c ON c.cliente_id = e.cliente_id
WHERE e.cliente_id IS NOT NULL
  AND c.cliente_id IS NULL
""")

,expediente_id,cliente_id,deuda_total
0,827,960,46388.00
1,721,995,48649.85
2,161,1096,38846.60
3,866,1148,49214.00
4,579,930,64710.00
5,613,1096,33732.00
6,523,1170,47547.21
7,267,987,63297.00
8,708,1298,14435.00
9,115,955,47538.00


Expedientes sin cliente asignado

In [23]:
q("SELECT COUNT(*) AS sin_cliente FROM expedientes WHERE cliente_id IS NULL")

,sin_cliente
0,10
